In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content
!rm -rf FireSmoke_YOLO11
!git clone https://github.com/SatyamDwivedi-sd/FireSmoke_YOLO11.git
%cd FireSmoke_YOLO11
!ls
!ls scripts

/content
Cloning into 'FireSmoke_YOLO11'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 116 (delta 50), reused 116 (delta 50), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 28.35 KiB | 14.18 MiB/s, done.
Resolving deltas: 100% (50/50), done.
/content/FireSmoke_YOLO11
AGENTS.md  data       PROJECT_PLAN.md  requirements.txt  videos
archive    models     README.md        results
configs    notebooks  reports	       scripts
01_check_dataset.py	     06_fixed_interval_video.py
02_prepare_dfire_dataset.py  07_adaptive_keyframe_video.py
03_train_yolo11n.py	     08_evaluate_methods.py
04_validate_model.py	     09_export_model.py
05_full_frame_video.py


In [ ]:
!ls -lh /content/drive/MyDrive/FireSmoke_YOLO11/archive.zip

-rw------- 1 root root 2.9G Apr 25 00:16 /content/drive/MyDrive/FireSmoke_YOLO11/archive.zip


In [ ]:
!mkdir -p data/raw/dfire
!unzip -q /content/drive/MyDrive/FireSmoke_YOLO11/archive.zip -d data/raw/dfire

In [ ]:
!find data/raw/dfire -maxdepth 3 -type d | sort | head -30

data/raw/dfire
data/raw/dfire/test
data/raw/dfire/test/images
data/raw/dfire/test/labels
data/raw/dfire/train
data/raw/dfire/train/images
data/raw/dfire/train/labels


In [ ]:
!python scripts/02_prepare_dfire_dataset.py \
  --raw-root data/raw/dfire \
  --output-root data/processed/fire_smoke_yolo11 \
  --valid-ratio 0.15 \
  --seed 42 \
  --overwrite

D-Fire dataset prepared successfully.
  Train images copied:   14638
  Valid images copied:   2583
  Test images copied:    4306
  Empty labels:          9841
  Invalid rows dropped:  8
  data.yaml:             data/processed/fire_smoke_yolo11/data.yaml


In [ ]:
!python scripts/01_check_dataset.py --dataset-root data/processed/fire_smoke_yolo11

Dataset root: data/processed/fire_smoke_yolo11

Split: train
  Total images:      14638
  Total label files: 14638
  Missing labels:    0
  Empty labels:      6686
  Invalid rows:      0
  Class counts:
    0 fire:  8088
    1 smoke: 10030

Split: valid
  Total images:      2583
  Total label files: 2583
  Missing labels:    0
  Empty labels:      1147
  Invalid rows:      0
  Class counts:
    0 fire:  1462
    1 smoke: 1784

Split: test
  Total images:      4306
  Total label files: 4306
  Missing labels:    0
  Empty labels:      2008
  Invalid rows:      0
  Class counts:
    0 fire:  2307
    1 smoke: 2878

Overall Summary
  Total images:      21527
  Total label files: 21527
  Missing labels:    0
  Empty labels:      9841
  Invalid rows:      0
  Class counts:
    0 fire:  11857
    1 smoke: 14692
  Total issues:      0


In [ ]:
!pip install -U ultralytics opencv-python pandas matplotlib pyyaml tqdm scikit-learn tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 150.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 170.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 170.9 MB/s eta 0:00:00
  Attempting uninstall: tabulate
    Found existing installation: tabulate 0.9.0
    Uninstalling tabulate-0.9.0:
      Successfully uninstalled tabulate-0.9.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
   

In [ ]:
import ultralytics
print(ultralytics.__version__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
8.4.41


In [ ]:
!python scripts/03_train_yolo11n.py \
  --data data/processed/fire_smoke_yolo11/data.yaml \
  --epochs 50 \
  --imgsz 640 \
  --batch 16 \
  --device 0 \
  --workers 2 \
  --project runs \
  --name exp01_yolo11n_dfire \
  --patience 15

Selected device: 0
Starting YOLO11n training
  data:     data/processed/fire_smoke_yolo11/data.yaml
  model:    yolo11n.pt
  epochs:   50
  imgsz:    640
  batch:    16
  device:   0
  workers:  2
  project:  runs
  name:     exp01_yolo11n_dfire
  patience: 15
  fraction: 1.0
Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/fire_smoke_yolo11/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, io

In [ ]:
!mkdir -p /content/drive/MyDrive/FireSmoke_YOLO11/models
!mkdir -p /content/drive/MyDrive/FireSmoke_YOLO11/runs

!cp runs/exp01_yolo11n_dfire/weights/best.pt /content/drive/MyDrive/FireSmoke_YOLO11/models/yolo11n_dfire_best.pt
!cp runs/exp01_yolo11n_dfire/weights/last.pt /content/drive/MyDrive/FireSmoke_YOLO11/models/yolo11n_dfire_last.pt

!cp -r runs/exp01_yolo11n_dfire /content/drive/MyDrive/FireSmoke_YOLO11/runs/

In [ ]:
!ls -lh /content/drive/MyDrive/FireSmoke_YOLO11/models
!ls -lh /content/drive/MyDrive/FireSmoke_YOLO11/runs/exp01_yolo11n_dfire

total 11M
-rw------- 1 root root 5.2M Apr 25 07:37 yolo11n_dfire_best.pt
-rw------- 1 root root 5.2M Apr 25 07:37 yolo11n_dfire_last.pt
total 5.7M
-rw------- 1 root root 1.7K Apr 25 07:37 args.yaml
-rw------- 1 root root 152K Apr 25 07:37 BoxF1_curve.png
-rw------- 1 root root 138K Apr 25 07:37 BoxP_curve.png
-rw------- 1 root root 141K Apr 25 07:37 BoxPR_curve.png
-rw------- 1 root root 150K Apr 25 07:37 BoxR_curve.png
-rw------- 1 root root 114K Apr 25 07:37 confusion_matrix_normalized.png
-rw------- 1 root root 112K Apr 25 07:37 confusion_matrix.png
-rw------- 1 root root 130K Apr 25 07:37 labels.jpg
-rw------- 1 root root 8.1K Apr 25 07:37 results.csv
-rw------- 1 root root 253K Apr 25 07:37 results.png
-rw------- 1 root root 602K Apr 25 07:37 train_batch0.jpg
-rw------- 1 root root 513K Apr 25 07:37 train_batch1.jpg
-rw------- 1 root root 530K Apr 25 07:37 train_batch2.jpg
-rw------- 1 root root 382K Apr 25 07:37 train_batch36600.jpg
-rw------- 1 root root 321K Apr 25 07:37 train_